In [ ]:
#| default_exp diff

# Text diffs

> Line and word differences between plain-text sources.

`diff_lines` pairs similar changed lines and identifies the words that were removed or inserted. Renderers can use the same result for terminal output or HTML.

In [ ]:
from difflib import unified_diff, SequenceMatcher
import re

In [ ]:
from fastcore.test import *
from fastcore.basics import PrettyString

In [ ]:
def _source_lines(old, new, context, final_newline):
    def lines(s): return s.splitlines(keepends=True) if final_newline else [l+'\n' for l in s.splitlines()]
    a,b = lines(old),lines(new)
    return list(unified_diff(a, b, n=max(len(a), len(b)) if context is None else context))

In [ ]:
def _word_diff(a, b, min_ratio):
    if a==b: return None
    a_nl,b_nl = a.endswith('\n'),b.endswith('\n')
    a,b = a.removesuffix('\n'),b.removesuffix('\n')
    if SequenceMatcher(None, re.findall(r'\w+', a), re.findall(r'\w+', b), autojunk=False).ratio()<min_ratio: return None
    ta,tb = (re.findall(r'\w+|\s+|[^\w\s]', s) for s in (a,b))
    ops = SequenceMatcher(lambda t: t.isspace(), ta, tb, autojunk=False).get_opcodes()
    def width(op): return max(sum(map(len, ta[op[1]:op[2]])), sum(map(len, tb[op[3]:op[4]])))
    res = []
    a0 = b0 = 0
    def changes(i,j):
        if a0<i: res.append(('-', ''.join(ta[a0:i])))
        if b0<j: res.append(('+', ''.join(tb[b0:j])))
    for i,(tag,i1,i2,j1,j2) in enumerate(ops):
        if tag!='equal': continue
        if 0<i<len(ops)-1 and width(ops[i])<min(width(ops[i-1]),width(ops[i+1])): continue
        changes(i1,j1)
        res.append((' ', ''.join(ta[i1:i2])))
        a0,b0 = i2,j2
    changes(len(ta),len(tb))
    if a_nl!=b_nl: res.append(('-' if a_nl else '+', '\n'))
    return res

In [ ]:
def _line_row(line): return line[0], [(' ', line[1:].removesuffix('\n'))]

def _refine(lines, min_ratio):
    i = 0
    while i<len(lines):
        if not lines[i].startswith('-'):
            yield _line_row(lines[i])
            i += 1
            continue
        j = i
        while j<len(lines) and lines[j].startswith('-'): j += 1
        k = j
        while k<len(lines) and lines[k].startswith('+'): k += 1
        olds,news = lines[i:j],lines[j:k]
        start = 0
        for p,(o,n) in enumerate(zip(olds,news)):
            w = _word_diff(o[1:], n[1:], min_ratio)
            if not w: continue
            for line in olds[start:p]+news[start:p]: yield _line_row(line)
            yield '!',w
            start = p+1
        for line in olds[start:]+news[start:]: yield _line_row(line)
        i = k

## Structured changes

In [ ]:
MIN_RATIO = .5

def diff_lines(
    old, # Original source string
    new, # New source string
    context=3, # Unchanged lines shown either side of a change; None shows the full source
    final_newline=True, # Distinguish a missing final newline (unnecessary for notebook cells)?
    min_ratio=MIN_RATIO # Minimum shared-word similarity for refining a changed line pair
): # List of (line marker, [(span marker, text), ...]) pairs
    "Line diffs with word-level replacements for similar changed lines."
    lines = _source_lines(old, new, context, final_newline)
    rows = list(_refine(lines[2:], min_ratio))
    if final_newline and old.endswith('\n')!=new.endswith('\n'):
        tag = '-' if old.endswith('\n') else '+'
        for kind,spans in reversed(rows):
            if kind==tag: spans.append((tag, '\n'))
            if kind in (tag, '!', ' ', '@'): break
    return rows

Each row has a marker and a list of spans. The row markers are ` ` for unchanged context, `-` for a deleted line, `+` for an added line, `!` for a refined replacement, and `@` for a hunk header. Adding the row marker to its spans reconstructs an ordinary unified-diff line; replacement spans instead combine old and new text.

Spans use ` ` for unchanged text, `-` for removed text and `+` for inserted text. When a paired line gains or loses its final newline, the change is a separate `('+', '\n')` or `('-', '\n')` span. The strings contain no display markup. Use `final_newline=False` to ignore this distinction for notebook cells:

In [ ]:
old = 'x=1\ny=2\nz=3\nk=4'
new = 'x = 100\ny=2\nw=5\nq=6'
rows = diff_lines(old, new, context=None, final_newline=False)
test_eq([kind for kind,_ in rows], ['@', '!', ' ', '-', '-', '+', '+'])
mixed = diff_lines('apple\nx = 1', 'pear\nx = 2\nextra', final_newline=False)[1:]
test_eq([kind for kind,_ in mixed], ['-', '+', '!', '+'])
rows

[('@', [(' ', '@ -1,4 +1,4 @@')]),
 ('!', [(' ', 'x'), ('+', ' '), (' ', '='), ('-', '1'), ('+', ' 100')]),
 (' ', [(' ', 'y=2')]),
 ('-', [(' ', 'z=3')]),
 ('-', [(' ', 'k=4')]),
 ('+', [(' ', 'w=5')]),
 ('+', [(' ', 'q=6')])]

A line pair is refined when its shared-word similarity reaches `min_ratio`. Unrelated lines remain separate deletions and additions. Words, punctuation and whitespace are separate tokens; whitespace does not anchor matches.

Short coincidental matches inside a larger rewrite are included in the replaced phrase. Longer unchanged text still separates independent edits:

In [ ]:
before = 'selects the fields, shown by the vectors'
after = 'reverses the coordinates, arranged in two columns'
suffix = '. Dragging either point redraws both points.'
phrase = diff_lines('The model '+before+suffix, 'The model '+after+suffix, final_newline=False)[1]
test_eq(phrase, ('!', [(' ', 'The model '), ('-', before), ('+', after), (' ', suffix)]))
phrase

('!',
 [(' ', 'The model '),
  ('-', 'selects the fields, shown by the vectors'),
  ('+', 'reverses the coordinates, arranged in two columns'),
  (' ', '. Dragging either point redraws both points.')])

In [ ]:
shared = ' while both points retain their positions '
separate = diff_lines('We select fields'+shared+'using the old formula.',
                      'We reverse coordinates'+shared+'using the new expression.', final_newline=False)[1]
test_eq(separate, ('!', [(' ', 'We '), ('-', 'select fields'), ('+', 'reverse coordinates'),
                       (' ', shared+'using the '), ('-', 'old formula'), ('+', 'new expression'), (' ', '.')]))
test_eq(diff_lines('hello\n', 'hello')[1:], [('!', [(' ', 'hello'), ('-', '\n')])])
test_eq(diff_lines('hello', 'hello\n')[1:], [('!', [(' ', 'hello'), ('+', '\n')])])
test_eq(diff_lines('hello\n', 'hello', final_newline=False), [])
separate

('!',
 [(' ', 'We '),
  ('-', 'select fields'),
  ('+', 'reverse coordinates'),
  (' ', ' while both points retain their positions using the '),
  ('-', 'old formula'),
  ('+', 'new expression'),
  (' ', '.')])

## Unified text

In [ ]:
def source_diff(
    old_source, # Original source string
    new_source, # New source string
    context=3, # Unchanged lines shown either side of a change
    final_newline=True # Distinguish a missing final newline (unnecessary for notebook cells)?
):
    "Unified source diff, including missing-final-newline markers."
    lines = _source_lines(old_source, new_source, context, final_newline)
    return '\n'.join(l.removesuffix('\n') + ('' if l.endswith('\n') else '\n\\ No newline at end of file') for l in lines)

`source_diff` returns an ordinary unified diff rather than inline word spans. It marks a missing final newline in files; `final_newline=False` ignores that distinction for notebook cells:

In [ ]:
patch = source_diff('hello\n', 'hello')
assert r'\ No newline at end of file' in patch
test_eq(source_diff('hello\n', 'hello', final_newline=False), '')
print(patch)

--- 
+++ 
@@ -1 +1 @@
-hello
+hello
\ No newline at end of file


## Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()